In [70]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
import keras
from keras import layers
import tensorflow
from rbf_keras import kmeans_initializer
from keras.optimizers import RMSprop
from rbf_keras import rbflayer
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt

In [71]:
heart_attack = pd.read_csv('./Heart Attack.csv')
heart_attack.head()

,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin,class
0,64,1,66,160,83,160.0,1.80,0.012,negative
1,21,1,94,98,46,296.0,6.75,1.060,positive
2,55,1,64,160,77,270.0,1.99,0.003,negative
3,64,1,70,120,55,270.0,13.87,0.122,positive
4,55,1,64,112,65,300.0,1.08,0.003,negative


In [72]:
labels = []
for item in heart_attack['class']:
    if item == 'positive':
        labels.append(1)
    else:
        labels.append(0)

labels = pd.Series(labels)

In [73]:
df = pd.concat([heart_attack[['age', 'gender', 'impluse', 'pressurehight', 'pressurelow', 'glucose', 'kcm', 'troponin']].copy(), labels.copy()], axis=1)

In [74]:
scaler = MinMaxScaler((0,1))
df['age'] = scaler.fit_transform(df[['age']])
df['gender'] = scaler.fit_transform(df[['gender']])
df['impluse'] = scaler.fit_transform(df[['impluse']])
df['pressurehight'] = scaler.fit_transform(df[['pressurehight']])
df['pressurelow'] = scaler.fit_transform(df[['pressurelow']])
df['glucose'] = scaler.fit_transform(df[['glucose']])
df['kcm'] = scaler.fit_transform(df[['kcm']])
df['troponin'] = scaler.fit_transform(df[['troponin']])

In [75]:
x = df.iloc[:, :-1]
y = df.iloc[:, 8]

In [76]:
x.head()

,age,gender,impluse,pressurehight,pressurelow,glucose,kcm,troponin
0,0.561798,1.0,0.042163,0.651934,0.387931,0.247036,0.004935,0.001068
1,0.078652,1.0,0.067828,0.309392,0.068966,0.515810,0.021453,0.102826
2,0.460674,1.0,0.040330,0.651934,0.336207,0.464427,0.005569,0.000194
3,0.561798,1.0,0.045830,0.430939,0.146552,0.464427,0.045212,0.011749
4,0.460674,1.0,0.040330,0.386740,0.232759,0.523715,0.002533,0.000194


In [77]:
print(y)

0       0
1       1
2       0
3       1
4       0
       ..
1314    0
1315    1
1316    1
1317    1
1318    1
Name: 0, Length: 1319, dtype: int64


In [83]:
# kfold = KFold(n_splits=2)
# histories = []

# for train_index, test_index in kfold.split(x):
#     X_train, X_test = x.iloc[train_index], x.iloc[test_index]
#     Y_train, Y_test = y.iloc[train_index], y.iloc[test_index]
X_train, X_test = x.iloc[:1000], x.iloc[1000:]
Y_train, Y_test = y.iloc[:1000], y.iloc[1000:]

rbfLayer = rbflayer.RBFLayer(2, 
                             initializer=kmeans_initializer.InitCentersKMeans(x), 
                             betas=1.0, 
                             input_shape=([8]))
    
model2 = keras.models.Sequential()
model2.add(rbfLayer)
model2.add(layers.Dense(1, activation='linear'))
model2.compile(optimizer=RMSprop(),
              loss='mae',
              metrics=['accuracy'])
    
model2.fit(X_train, Y_train, epochs=50, batch_size=1)

c:\Users\colit\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


Epoch 1/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5228 - accuracy: 0.4660
Epoch 2/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.4167 - accuracy: 0.6120
Epoch 3/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3909 - accuracy: 0.6130
Epoch 4/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3901 - accuracy: 0.6130
Epoch 5/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3883 - accuracy: 0.6130
Epoch 6/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3869 - accuracy: 0.6130
Epoch 7/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3871 - accuracy: 0.6130
Epoch 8/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3870 - accuracy: 0.6130
Epoch 9/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.3867 - accuracy: 0.6130
Epoch 10/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.386

In [84]:
model2.summary()

Model: "sequential_11"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rbf_layer_11 (RBFLayer)     (None, 2)                 18        
                                                                 
 dense_11 (Dense)            (None, 1)                 3         
                                                                 
Total params: 21 (84.00 Byte)
Trainable params: 21 (84.00 Byte)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [85]:
loss, accuracy = model2.evaluate(X_test, Y_test, batch_size=1)
print("Loss: " + str(loss) + " %  ||   Acuratete: " + str(accuracy * 100) + " %")

319/319 [==============================] - 1s 2ms/step - loss: 0.3679 - accuracy: 0.6520
Loss: 0.3678814768791199 %  ||   Acuratete: 65.20376205444336 %


In [93]:
rbfLayer_10 = rbflayer.RBFLayer(1318, 
                             initializer=kmeans_initializer.InitCentersKMeans(x), 
                             betas=1.0, 
                             input_shape=([8]))
    
model_10 = keras.models.Sequential()
model_10.add(rbfLayer_10)
model_10.add(layers.Dense(1, activation='linear'))
model_10.compile(optimizer=RMSprop(),
              loss='mae',
              metrics=['accuracy'])
    
model_10.fit(X_train, Y_train, epochs=50, batch_size=1)

c:\Users\colit\AppData\Local\Programs\Python\Python39\lib\site-packages\sklearn\cluster\_kmeans.py:870: FutureWarning: The default value of `n_init` will change from 10 to 'auto' in 1.4. Set the value of `n_init` explicitly to suppress the warning
  warnings.warn(


Epoch 1/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.6496 - accuracy: 0.5600
Epoch 2/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.6136 - accuracy: 0.5850
Epoch 3/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.6084 - accuracy: 0.5840
Epoch 4/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5625 - accuracy: 0.6300
Epoch 5/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5896 - accuracy: 0.5720
Epoch 6/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5542 - accuracy: 0.5880
Epoch 7/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5285 - accuracy: 0.6040
Epoch 8/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5294 - accuracy: 0.6130
Epoch 9/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.5093 - accuracy: 0.6090
Epoch 10/50
1000/1000 [==============================] - 2s 2ms/step - loss: 0.501

In [94]:
model_10.summary()

Model: "sequential_15"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rbf_layer_15 (RBFLayer)     (None, 1318)              11862     
                                                                 
 dense_15 (Dense)            (None, 1)                 1319      
                                                                 
Total params: 13181 (51.49 KB)
Trainable params: 13181 (51.49 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [95]:
loss_10, accuracy_10 = model_10.evaluate(X_test, Y_test, batch_size=1)
print("Loss: " + str(loss_10) + " %  ||   Acuratete: " + str(accuracy_10 * 100) + " %")

319/319 [==============================] - 1s 2ms/step - loss: 0.4181 - accuracy: 0.6364
Loss: 0.41812488436698914 %  ||   Acuratete: 63.63636255264282 %
